In [1]:
import sys
import os

# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema
import pandas as pd

from pyspark.sql.functions import udtf
from pyspark.sql.types import StructType, StructField, StringType

In [3]:
@pandas_udf(get_subject_schema(), PandasUDFType.GROUPED_MAP)
def generate_subjects_udtf(pdf):
    # Read the participants.tsv file
    participantsInfo = pd.read_table('../ds004504/participants.tsv')
    
    # Filter the subjects by group and prepare the data
    subjects = []
    for group, group_name in [("A", "GroupA"), ("C", "GroupC"), ("F", "GroupD")]:
        group_subjects = participantsInfo[participantsInfo["Group"] == group]["participant_id"].tolist()
        subjects.extend([{"SubjectID": sub, "Group": group_name} for sub in group_subjects])
    
    # Return a DataFrame with the subjects and their groups
    return pd.DataFrame(subjects)

In [7]:
spark = SparkSession.builder \
    .appName("SubjectPopulation") \
    .getOrCreate()

# Define the schema explicitly
schema = get_subject_schema()

print(schema)


StructType([StructField('SubjectID', StringType(), False), StructField('Group', StringType(), False)])


In [12]:
subjects_df = spark.createDataFrame([("dummy",)], ["Group"])


In [13]:
subject_df = subjects_df.groupby("key").apply(generate_subjects_udtf)
subjects_df.printSchema()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `key` cannot be resolved. Did you mean one of the following? [`Group`].;
'Project ['key, Group#8]
+- LogicalRDD [Group#8], false


In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
import pandas as pd

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SubjectPopulation") \
    .getOrCreate()

# Define the schema
def get_subject_schema():
    return StructType([
        StructField("SubjectID", StringType(), False),
        StructField("Group", StringType(), False),
    ])

# Read the .tsv file with pandas
participantsInfo = pd.read_table('../ds004504/participants.tsv')

# Convert pandas DataFrame to Spark DataFrame
subjects_df = spark.createDataFrame(participantsInfo)

# Ensure the columns match the expected schema
subjects_df = subjects_df.select(
    subjects_df["participant_id"].alias("SubjectID"),
    subjects_df["Group"]
)

# Show the schema and initial data
subjects_df.printSchema()
subjects_df.show()

# If you need to apply a UDF to transform or generate additional data
def generate_additional_info(subject_id, group):
    # Example transformation or additional data generation
    return f"Processed: {subject_id}, {group}"

generate_info_udf = udf(generate_additional_info, StringType())

# Apply the UDF to create a new column
subjects_df = subjects_df.withColumn("AdditionalInfo", generate_info_udf("SubjectID", "Group"))

# Show the results
subjects_df.show()

root
 |-- SubjectID: string (nullable = true)
 |-- Group: string (nullable = true)



+---------+-----+
|SubjectID|Group|
+---------+-----+
|  sub-001|    A|
|  sub-002|    A|
|  sub-003|    A|
|  sub-004|    A|
|  sub-005|    A|
|  sub-006|    A|
|  sub-007|    A|
|  sub-008|    A|
|  sub-009|    A|
|  sub-010|    A|
|  sub-011|    A|
|  sub-012|    A|
|  sub-013|    A|
|  sub-014|    A|
|  sub-015|    A|
|  sub-016|    A|
|  sub-017|    A|
|  sub-018|    A|
|  sub-019|    A|
|  sub-020|    A|
+---------+-----+
only showing top 20 rows



+---------+-----+--------------------+
|SubjectID|Group|      AdditionalInfo|
+---------+-----+--------------------+
|  sub-001|    A|Processed: sub-00...|
|  sub-002|    A|Processed: sub-00...|
|  sub-003|    A|Processed: sub-00...|
|  sub-004|    A|Processed: sub-00...|
|  sub-005|    A|Processed: sub-00...|
|  sub-006|    A|Processed: sub-00...|
|  sub-007|    A|Processed: sub-00...|
|  sub-008|    A|Processed: sub-00...|
|  sub-009|    A|Processed: sub-00...|
|  sub-010|    A|Processed: sub-01...|
|  sub-011|    A|Processed: sub-01...|
|  sub-012|    A|Processed: sub-01...|
|  sub-013|    A|Processed: sub-01...|
|  sub-014|    A|Processed: sub-01...|
|  sub-015|    A|Processed: sub-01...|
|  sub-016|    A|Processed: sub-01...|
|  sub-017|    A|Processed: sub-01...|
|  sub-018|    A|Processed: sub-01...|
|  sub-019|    A|Processed: sub-01...|
|  sub-020|    A|Processed: sub-02...|
+---------+-----+--------------------+
only showing top 20 rows



In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SubjectPopulation") \
    .getOrCreate()

# Read the .tsv file directly into a Spark DataFrame
# Note: Use 'csv' format with tab delimiter for .tsv files
subjects_df = spark.read.csv(
    path='../ds004504/participants.tsv',
    sep='\t',  # Tab delimiter for .tsv files
    header=True,  # Assuming the first row is header
    inferSchema=True  # Automatically infer the schema
)

# Ensure the columns match the expected schema
# Rename columns if necessary
subjects_df = subjects_df.select(
    col("participant_id").alias("SubjectID"),
    col("Group")
)

# Show the schema and initial data
subjects_df.printSchema()
subjects_df.show()

# # If you need to add or transform columns, use Spark SQL functions
# # For example, adding a new column based on existing ones
# from pyspark.sql.functions import concat, lit

# subjects_df = subjects_df.withColumn(
#     "AdditionalInfo", 
#     concat(col("SubjectID"), lit("_"), col("Group"))
# )

# Show the results
subjects_df.show()

root
 |-- SubjectID: string (nullable = true)
 |-- Group: string (nullable = true)

+---------+-----+
|SubjectID|Group|
+---------+-----+
|  sub-001|    A|
|  sub-002|    A|
|  sub-003|    A|
|  sub-004|    A|
|  sub-005|    A|
|  sub-006|    A|
|  sub-007|    A|
|  sub-008|    A|
|  sub-009|    A|
|  sub-010|    A|
|  sub-011|    A|
|  sub-012|    A|
|  sub-013|    A|
|  sub-014|    A|
|  sub-015|    A|
|  sub-016|    A|
|  sub-017|    A|
|  sub-018|    A|
|  sub-019|    A|
|  sub-020|    A|
+---------+-----+
only showing top 20 rows

+---------+-----+--------------+
|SubjectID|Group|AdditionalInfo|
+---------+-----+--------------+
|  sub-001|    A|     sub-001_A|
|  sub-002|    A|     sub-002_A|
|  sub-003|    A|     sub-003_A|
|  sub-004|    A|     sub-004_A|
|  sub-005|    A|     sub-005_A|
|  sub-006|    A|     sub-006_A|
|  sub-007|    A|     sub-007_A|
|  sub-008|    A|     sub-008_A|
|  sub-009|    A|     sub-009_A|
|  sub-010|    A|     sub-010_A|
|  sub-011|    A|     sub-011

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SubjectPopulation") \
    .getOrCreate()

# Define the schema function
def get_subject_schema():
    return StructType([
        StructField("SubjectID", StringType(), False),
        StructField("Group", StringType(), False),
    ])

# Read the .tsv file directly into a Spark DataFrame with the defined schema
subjects_df = spark.read.csv(
    path='../ds004504/participants.tsv',
    sep='\t',  # Tab delimiter for .tsv files
    header=True,  # Assuming the first row is header
    schema=get_subject_schema()  # Explicitly define the schema
)

# Show the schema to verify
subjects_df.printSchema()

# Show the data
subjects_df.show()

# If you need to add or transform columns, use Spark SQL functions
from pyspark.sql.functions import concat, lit

subjects_df = subjects_df.withColumn(
    "AdditionalInfo", 
    concat(col("SubjectID"), lit("_"), col("Group"))
)

# Show the results
subjects_df.show()

root
 |-- SubjectID: string (nullable = true)
 |-- Group: string (nullable = true)

+---------+-----+
|SubjectID|Group|
+---------+-----+
|  sub-001|    F|
|  sub-002|    F|
|  sub-003|    M|
|  sub-004|    F|
|  sub-005|    M|
|  sub-006|    F|
|  sub-007|    F|
|  sub-008|    M|
|  sub-009|    F|
|  sub-010|    M|
|  sub-011|    M|
|  sub-012|    M|
|  sub-013|    F|
|  sub-014|    M|
|  sub-015|    M|
|  sub-016|    F|
|  sub-017|    F|
|  sub-018|    F|
|  sub-019|    F|
|  sub-020|    M|
+---------+-----+
only showing top 20 rows

+---------+-----+--------------+
|SubjectID|Group|AdditionalInfo|
+---------+-----+--------------+
|  sub-001|    F|     sub-001_F|
|  sub-002|    F|     sub-002_F|
|  sub-003|    M|     sub-003_M|
|  sub-004|    F|     sub-004_F|
|  sub-005|    M|     sub-005_M|
|  sub-006|    F|     sub-006_F|
|  sub-007|    F|     sub-007_F|
|  sub-008|    M|     sub-008_M|
|  sub-009|    F|     sub-009_F|
|  sub-010|    M|     sub-010_M|
|  sub-011|    M|     sub-011

25/03/27 14:59:08 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 5, schema size: 2
CSV file: file:///Users/user/eeg-ds004504/ds004504/participants.tsv
25/03/27 14:59:08 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 5, schema size: 2
CSV file: file:///Users/user/eeg-ds004504/ds004504/participants.tsv


In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SubjectPopulation") \
    .getOrCreate()

# Define the schema function with the correct column names from the .tsv file
def get_subject_schema():
    return StructType([
        StructField("participant_id", StringType(), False),
        StructField("Gender", StringType(), False),
        StructField("Age", IntegerType(), False),
        StructField("Group", StringType(), False),
        StructField("MMSE", IntegerType(), False),
    ])

# Read the .tsv file directly into a Spark DataFrame with the defined schema
subjects_df = spark.read.csv(
    path='../ds004504/participants.tsv',
    sep='\t',  # Tab delimiter for .tsv files
    header=True,  # Assuming the first row is header
    schema=get_subject_schema()  # Explicitly define the schema
)

# Show the schema to verify
subjects_df.printSchema()

# Show the data
subjects_df.show()

# If you need to rename or transform columns, use Spark SQL functions
from pyspark.sql.functions import col

# For example, if you want to rename 'participant_id' to 'SubjectID'
subjects_df = subjects_df.select(
    col("participant_id").alias("SubjectID"),
    col("Gender"),
    col("Age"),
    col("Group"),
    col("MMSE")
)

# Show the results after renaming
subjects_df.show()

root
 |-- participant_id: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Group: string (nullable = true)
 |-- MMSE: integer (nullable = true)

+--------------+------+---+-----+----+
|participant_id|Gender|Age|Group|MMSE|
+--------------+------+---+-----+----+
|       sub-001|     F| 57|    A|  16|
|       sub-002|     F| 78|    A|  22|
|       sub-003|     M| 70|    A|  14|
|       sub-004|     F| 67|    A|  20|
|       sub-005|     M| 70|    A|  22|
|       sub-006|     F| 61|    A|  14|
|       sub-007|     F| 79|    A|  20|
|       sub-008|     M| 62|    A|  16|
|       sub-009|     F| 77|    A|  23|
|       sub-010|     M| 69|    A|  20|
|       sub-011|     M| 71|    A|  22|
|       sub-012|     M| 63|    A|  18|
|       sub-013|     F| 64|    A|  20|
|       sub-014|     M| 77|    A|  14|
|       sub-015|     M| 61|    A|  18|
|       sub-016|     F| 68|    A|  14|
|       sub-017|     F| 61|    A|   6|
|       sub-018|    

In [19]:

def generate_subjects_info(spark, filePath='../ds004504/participants.tsv'):
    participantsInfo = pd.read_table(filePath)
    # Convert pandas DataFrame to Spark DataFrame with the defined schema
    subjects_df = spark.createDataFrame(participantsInfo)
    

    # Show the schema to verify
    subjects_df.printSchema()
    
    # Show the data
    subjects_df.show()
    
    
    subjects_df = subjects_df.select(
        col("participant_id").alias("SubjectID"),
        col("Group")
    )


    return subjects_df
